In [120]:
import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd
import numpy as np

import ast

import ujson
import re

In [121]:
#----------------- TASK 0: EXTRACT -----------------
# Leer dataset con pandas
df = pd.read_csv('../data/listings.csv', low_memory=False)

print("==================================================")
print("   INICIANDO PREPROCESAMIENTO DEL DATASET AIRBNB  ")
print("==================================================\n")

#----------------- TASK 1: TRANSFORM -----------------

# =====================================================================
# 1. Limpiar e imputar la variable 'bathrooms' desde 'bathrooms_text'
# =====================================================================
print("[PASO 1] Procesando la variable 'bathrooms' y 'bathrooms_text'...")

df['bathrooms'] = pd.to_numeric(df['bathrooms'], errors='coerce')

def parse_bathrooms_text(text):
    if pd.isna(text): return np.nan
    text = text.lower().strip()
    if 'half' in text or 'medio' in text: 
        return 0.5
    match = re.search(r'[\d\.]+', text)
    return float(match.group()) if match else np.nan

df['bathrooms_text_num'] = df['bathrooms_text'].apply(parse_bathrooms_text)

mask_mismatch = (
    df['bathrooms'].notna() & 
    df['bathrooms_text_num'].notna() & 
    (df['bathrooms'] != df['bathrooms_text_num'])
)
mismatch = mask_mismatch.sum()

print(f" -> Discrepancias encontradas entre columnas: {mismatch}")
if mismatch > 0:
    # Mostramos un par de ejemplos de las discrepancias en el log si existen
    print(" -> Ejemplos de discrepancias:")
    print(df[mask_mismatch][['bathrooms', 'bathrooms_text', 'bathrooms_text_num']].head())

if mismatch / df.shape[0] < 0.1:
    print(f" -> Cambiando la variable 'bathrooms' por 'bathrooms_text_num' (discrepancia < 10%)\n")
    df['bathrooms'] = df['bathrooms_text_num']

# =====================================================================
# 2. Imputar manualmente 5 valores faltantes de 'bathrooms'
# =====================================================================
print("[PASO 2] Imputando valores nulos específicos de 'bathrooms'...")

ids_imputar = [907846578011191658, 1071778548709501720, 1149444780882723841, 1199390966969605315, 1209436626693592917]

# Verificamos si existen y son nulos antes de imputar
if (df.loc[df['id'].isin(ids_imputar), 'bathrooms'].isnull()).all():
    df.loc[df['id'] == 907846578011191658, 'bathrooms'] = 1
    df.loc[df['id'].isin([1071778548709501720, 1149444780882723841, 1199390966969605315, 1209436626693592917]), 'bathrooms'] = 0
    print(" -> 5 valores nulos imputados con éxito.\n")
else:
    print(" -> AVISO: Los 5 IDs específicos no son nulos o no se encontraron todos en el DataFrame.\n")

# =====================================================================
# 3. Crear variable binaria 'bathroom_shared'
# =====================================================================
print("[PASO 3] Creando variable binaria 'bathroom_shared'...")

df['bathroom_shared'] = df['bathrooms_text'].str.contains('shared|compartido', case=False, na=False)
print(f" -> Variable creada. Total de baños compartidos detectados: {df['bathroom_shared'].sum()}\n")

# =====================================================================
# 4 & 5. Imputar nulos en la variable 'bedrooms'
# =====================================================================
print("[PASO 4 y 5] Imputando variable 'bedrooms'...")
print(f" -> Nulos iniciales en bedrooms: {df['bedrooms'].isnull().sum()}")

mask_room = df['room_type'].isin(['Shared room', 'Private room'])
mask_bedrooms_null = df['bedrooms'].isnull()

df.loc[mask_room & mask_bedrooms_null, 'bedrooms'] = 1
print(f" -> Nulos tras aplicar regla de Shared/Private room: {df['bedrooms'].isnull().sum()}")

df['bedrooms'] = df['bedrooms'].fillna(
    df.groupby('accommodates')['bedrooms'].transform('median')
)
print(f" -> Nulos tras imputar por mediana de accommodates: {df['bedrooms'].isnull().sum()}\n")

# =====================================================================
# 6. Imputar nulos en la variable 'beds'
# =====================================================================
print("[PASO 6] Imputando variable 'beds'...")
print(f" -> Nulos iniciales en beds: {df['beds'].isnull().sum()}")

df['beds'] = df['beds'].fillna(
    df.groupby('accommodates')['beds'].transform('median')
)
print(f" -> Nulos tras imputar por mediana de accommodates: {df['beds'].isnull().sum()}\n")

# =====================================================================
# 7 & 8. Procesar 'amenities' a JSON y crear 'num_amenities'
# =====================================================================
print("[PASO 7 y 8] Procesando 'amenities' con ujson...")

def parse_amenities(amenities_str):
    if pd.isna(amenities_str):
        return []
    try:
        json_str = amenities_str.replace("'", '"')
        return ujson.loads(json_str)
    except:
        return [] 

df['amenities'] = df['amenities'].apply(parse_amenities)
df['num_amenities'] = df['amenities'].apply(len)

print(f" -> Extracción completada. Promedio de amenities por anuncio: {df['num_amenities'].mean():.2f}\n")

# =====================================================================
# 9. Limpiar formato de la variable 'price'
# =====================================================================
print("[PASO 9] Limpiando formato de 'price'...")
# Mostramos un ejemplo antes de limpiar para comprobar el formato
ejemplo_precio_antes = df['price'].dropna().iloc[0]

df['price'] = (
    df['price'].astype(str)
    .replace(r'[\$,%€]', '', regex=True)  
    .replace(r',', '', regex=True)        
    .astype(float)                        
)
print(f" -> Ejemplo antes: {ejemplo_precio_antes} | Ejemplo después: {df['price'].dropna().iloc[0]}\n")

# =====================================================================
# 10. Imputar manualmente el precio anómalo (ID: 47444051)
# =====================================================================
print("[PASO 10] Corrigiendo precio anómalo (ID 47444051)...")
if 47444051 in df['id'].values:
    precio_previo = df.loc[df['id'] == 47444051, 'price'].values[0]
    df.loc[df['id'] == 47444051, 'price'] = 182.0
    precio_nuevo = df.loc[df['id'] == 47444051, 'price'].values[0]
    print(f" -> Precio corregido: de {precio_previo} a {precio_nuevo}\n")
else:
    print(" -> AVISO: No se encontró el ID 47444051 en este DataFrame.\n")

# =====================================================================
# 11. Transformación logarítmica para 'price'
# =====================================================================
print("[PASO 11] Aplicando transformación logarítmica al precio...")
df['price_log'] = np.log1p(df['price'])
print(f" -> Creada columna 'price_log'. Rango actual: Min {df['price_log'].min():.2f} - Max {df['price_log'].max():.2f}\n")

# =====================================================================
# 12. Preprocesamiento de 'description' para PLN
# =====================================================================
print("[PASO 12] Limpiando textos de 'description'...")

def clean_text_nlp(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['description'] = df['description'].apply(clean_text_nlp)
print(f" -> Textos limpios. Ejemplo de los primeros 30 caracteres procesados:\n    '{df['description'].dropna().iloc[0][:30]}...'\n")

# =====================================================================
# 13. Codificación One-Hot (OHE) para 'room_type'
# =====================================================================
print("[PASO 13] Aplicando One-Hot Encoding a 'room_type'...")
cols_antes = len(df.columns)

df = pd.get_dummies(df, columns=['room_type'], prefix='room', drop_first=False)

cols_nuevas = [col for col in df.columns if col.startswith('room_')]
print(f" -> Columnas generadas ({len(cols_nuevas)}): {', '.join(cols_nuevas)}")
print(f" -> Total de columnas en el DataFrame: de {cols_antes} a {len(df.columns)}\n")

print("==================================================")
print("            PREPROCESAMIENTO FINALIZADO           ")
print("==================================================")

   INICIANDO PREPROCESAMIENTO DEL DATASET AIRBNB  

[PASO 1] Procesando la variable 'bathrooms' y 'bathrooms_text'...
 -> Discrepancias encontradas entre columnas: 1
 -> Ejemplos de discrepancias:
      bathrooms bathrooms_text  bathrooms_text_num
5523        1.0      1.5 baths                 1.5
 -> Cambiando la variable 'bathrooms' por 'bathrooms_text_num' (discrepancia < 10%)

[PASO 2] Imputando valores nulos específicos de 'bathrooms'...
 -> 5 valores nulos imputados con éxito.

[PASO 3] Creando variable binaria 'bathroom_shared'...
 -> Variable creada. Total de baños compartidos detectados: 684

[PASO 4 y 5] Imputando variable 'bedrooms'...
 -> Nulos iniciales en bedrooms: 181
 -> Nulos tras aplicar regla de Shared/Private room: 77
 -> Nulos tras imputar por mediana de accommodates: 0

[PASO 6] Imputando variable 'beds'...
 -> Nulos iniciales en beds: 894
 -> Nulos tras imputar por mediana de accommodates: 0

[PASO 7 y 8] Procesando 'amenities' con ujson...
 -> Extracción complet

In [122]:
print("==================================================")
print("   INICIANDO PREPROCESAMIENTO: VARIABLES DE HOST  ")
print("==================================================\n")

# =====================================================================
# 1. Convertir 'host_since' a formato de fecha
# =====================================================================
print("[PASO 1] Convirtiendo 'host_since' a datetime...")
nulos_fechas_antes = df['host_since'].isnull().sum()
df['host_since'] = pd.to_datetime(df['host_since'], errors='coerce')
nulos_fechas_despues = df['host_since'].isnull().sum()
print(f" -> Nulos antes: {nulos_fechas_antes} | Nulos después de la conversión: {nulos_fechas_despues}")
if nulos_fechas_despues > nulos_fechas_antes:
    print(f" -> AVISO: {nulos_fechas_despues - nulos_fechas_antes} valores no tenían un formato de fecha válido y se han forzado a nulo.\n")
else:
    print(" -> Conversión completada sin errores de formato.\n")

# =====================================================================
# 2. Convertir tasas a float (host_acceptance_rate, host_response_rate)
# =====================================================================
print("[PASO 2] Convirtiendo tasas (rates) a numérico (quitando '%')...")
for col in ['host_acceptance_rate', 'host_response_rate']:
    df[col] = df[col].astype(str).str.replace('%', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')
    
print(f" -> 'host_acceptance_rate' - Nulos: {df['host_acceptance_rate'].isnull().sum()} | Rango actual: [{df['host_acceptance_rate'].min()}, {df['host_acceptance_rate'].max()}]")
print(f" -> 'host_response_rate'   - Nulos: {df['host_response_rate'].isnull().sum()} | Rango actual: [{df['host_response_rate'].min()}, {df['host_response_rate'].max()}]\n")

# =====================================================================
# 3. Pasar a formato booleano las variables principales
# =====================================================================
print("[PASO 3] Convirtiendo a booleanos (superhost, profile_pic, identity)...")
bool_cols = ['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified']
for col in bool_cols:
    df[col] = df[col].map({'t': True, 'f': False, True: True, False: False})
    # Traza para ver cómo queda la distribución inicial (ignorando nulos por ahora)
    distribucion = df[col].value_counts(dropna=False).to_dict()
    print(f" -> {col}: {distribucion}")
print(" -> Transformación booleana aplicada correctamente.\n")

# =====================================================================
# 4. Imputar valores nulos de 'host_is_superhost'
# =====================================================================
print("[PASO 4] Imputando 'host_is_superhost' basado en heurística de host...")
nulos_superhost_antes = df['host_is_superhost'].isnull().sum()

# Calculamos las métricas reales a nivel de host (sumando y promediando sus anuncios)
total_reviews_host = df.groupby('host_id')['number_of_reviews'].transform('sum')

if 'review_scores_rating' in df.columns:
    avg_rating_host = df.groupby('host_id')['review_scores_rating'].transform('mean')
else:
    avg_rating_host = 0

# Recreamos la lógica aproximada de Airbnb evaluando al HOST, no al anuncio
criterio_superhost = (
    (total_reviews_host >= 10) & 
    (df['host_response_rate'] >= 90) & 
    (df['host_acceptance_rate'] >= 90) & 
    (avg_rating_host >= 4.8)
)

df['host_is_superhost'] = df['host_is_superhost'].fillna(criterio_superhost)
print(f" -> Nulos imputados: {nulos_superhost_antes}")
print(f" -> Distribución final de 'host_is_superhost': \n{df['host_is_superhost'].value_counts().to_string()}\n")

# =====================================================================
# 5. Binarizar e imputar 'host_response_time'
# =====================================================================
print("[PASO 5] Binarizando e imputando 'host_response_time'...")
nulos_rt_antes = df['host_response_time'].isnull().sum()

# 1 para "menos de 1 hora", 0 para el resto
mapa_rt = {
    'within an hour': 1,
    'within a few hours': 0,
    'within a day': 0,
    'a few days or more': 0
}
df['host_response_time_bin'] = df['host_response_time'].map(mapa_rt)

def imputar_moda(x):
    if not x.mode().empty:
        return x.fillna(x.mode()[0])
    return x

df['host_response_time_bin'] = df.groupby('host_is_superhost')['host_response_time_bin'].transform(imputar_moda)
print(f" -> Nulos iniciales: {nulos_rt_antes}. Nulos tras imputar moda agrupada: {df['host_response_time_bin'].isnull().sum()}")
print(f" -> Distribución: 1 (Menos de 1 hora) = {sum(df['host_response_time_bin']==1)} | 0 (Más de 1 hora) = {sum(df['host_response_time_bin']==0)}\n")

# =====================================================================
# 6. Imputar tasas con VARIABILIDAD y CLIPPING (0 a 100)
# =====================================================================
print("[PASO 6] Imputando 'host_acceptance_rate' y 'host_response_rate' con variabilidad (ruido normal)...")

def imputar_con_variabilidad(df_target, col, agrupacion):
    nulos = df_target[col].isnull()
    
    # Calculamos la mediana y la desviación estándar por grupo
    mediana_grupo = df_target.groupby(agrupacion)[col].transform('median')
    std_grupo = df_target.groupby(agrupacion)[col].transform('std')
    
    # Si algún grupo tiene solo 1 elemento, std será NaN. Le asignamos un 5% de variabilidad por defecto.
    std_grupo = std_grupo.fillna(5.0)
    
    # Generamos ruido normal basado en la desviación estándar del grupo
    np.random.seed(42) # Para reproducibilidad
    ruido = np.random.normal(loc=0.0, scale=std_grupo)
    
    # Sumamos ruido a la mediana y aplicamos un CLIP estricto entre 0 y 100
    valores_propuestos = (mediana_grupo + ruido).clip(lower=0, upper=100)
    
    # Aplicamos solo a los nulos
    df_target.loc[nulos, col] = valores_propuestos[nulos]
    
    # Si ha quedado algún rezagado (ej. grupo entero era nulo), rellenamos con la mediana global
    df_target[col] = df_target[col].fillna(df_target[col].median())
    
    return nulos.sum(), df_target[col].mean()

# Aplicamos la función a ambas columnas
nulos_acc, media_acc = imputar_con_variabilidad(df, 'host_acceptance_rate', 'host_is_superhost')
nulos_res, media_res = imputar_con_variabilidad(df, 'host_response_rate', ['host_is_superhost', 'host_response_time_bin'])

print(f" -> 'host_acceptance_rate': {nulos_acc} valores imputados (Mediana + Ruido | Clip 0-100). Media final: {media_acc:.2f}%")
print(f" -> 'host_response_rate':   {nulos_res} valores imputados (Mediana + Ruido | Clip 0-100). Media final: {media_res:.2f}%\n")

# =====================================================================
# 7. Codificar con OHE 'host_verifications'
# =====================================================================
print("[PASO 7] Aplicando One-Hot Encoding a 'host_verifications'...")

df['verifications_clean'] = df['host_verifications'].astype(str).str.replace(r"[\[\]\']", "", regex=True)
verifications_dummies = df['verifications_clean'].str.get_dummies(sep=', ')
verifications_dummies = verifications_dummies.add_prefix('verification_')

df = pd.concat([df, verifications_dummies], axis=1)
print(f" -> Se han extraído {verifications_dummies.shape[1]} métodos de verificación.")
print(f" -> Top 3 métodos usados: \n{verifications_dummies.sum().sort_values(ascending=False).head(3).to_string()}\n")

# =====================================================================
# 8. Variable 'calculated_host_listings_hotel_rooms'
# =====================================================================
print("[PASO 8] Calculando 'calculated_host_listings_hotel_rooms'...")

df['calculated_host_listings_hotel_rooms'] = (
    df['calculated_host_listings_count'] - (
        df.get('calculated_host_listings_count_entire_homes', 0) + 
        df.get('calculated_host_listings_count_private_rooms', 0) + 
        df.get('calculated_host_listings_count_shared_rooms', 0)
    )
).clip(lower=0)

num_hoteles = (df['calculated_host_listings_hotel_rooms'] > 0).sum()
print(f" -> Habitaciones de hotel inferidas. Hay {num_hoteles} anuncios vinculados a anfitriones con habitaciones de hotel.\n")

# =====================================================================
# 9. Variable booleana 'company'
# =====================================================================
print("[PASO 9] Creando la variable booleana 'company'...")

keywords = r'\b(sl|s\.l\.|s\.a\.|sa|company|management|group|rentals|apartments|homes|villas|properties|gestión|holidays)\b'
mask_name = df['host_name'].astype(str).str.contains(keywords, case=False, regex=True, na=False)
mask_count = df['calculated_host_listings_count'] > 5

df['company'] = (mask_name | mask_count).astype(int)
df['company'] = (mask_name).astype(int)
print(f" -> Perfiles particulares: {sum(df['company']==0)} | Perfiles de empresa: {sum(df['company']==1)}\n")

# =====================================================================
# 10. Categorizar 'host_location' y crear 'host_near'
# =====================================================================
print("[PASO 10] Categorizando 'host_location' y creando 'host_near'...")

df['host_location'] = df['host_location'].fillna('Unknown').astype(str).str.lower()

cond_malaga = df['host_location'].str.contains('málaga|malaga')
cond_espana = df['host_location'].str.contains('españa|spain|madrid|barcelona|andalucía|andalucia')

df['host_location'] = np.select(
    [cond_malaga, cond_espana, df['host_location'] == 'unknown'], 
    ['Málaga', 'España', 'Desconocido'], 
    default='Internacional'
)

df['host_near'] = (df['host_location'] == 'Málaga').astype(int)

print(f" -> Distribución final de 'host_location':")
print(df['host_location'].value_counts().to_string())
print(f" -> % de anfitriones locales (Málaga): {(df['host_near'].mean() * 100):.2f}%\n")

print("==================================================")
print("     PREPROCESAMIENTO DE HOSTS FINALIZADO         ")
print("==================================================")

   INICIANDO PREPROCESAMIENTO: VARIABLES DE HOST  

[PASO 1] Convirtiendo 'host_since' a datetime...
 -> Nulos antes: 0 | Nulos después de la conversión: 0
 -> Conversión completada sin errores de formato.

[PASO 2] Convirtiendo tasas (rates) a numérico (quitando '%')...
 -> 'host_acceptance_rate' - Nulos: 643 | Rango actual: [0.0, 100.0]
 -> 'host_response_rate'   - Nulos: 929 | Rango actual: [0.0, 100.0]

[PASO 3] Convirtiendo a booleanos (superhost, profile_pic, identity)...
 -> host_is_superhost: {False: 6691, True: 2758, nan: 265}
 -> host_has_profile_pic: {True: 9487, False: 227}
 -> host_identity_verified: {True: 9167, False: 547}
 -> Transformación booleana aplicada correctamente.

[PASO 4] Imputando 'host_is_superhost' basado en heurística de host...
 -> Nulos imputados: 265
 -> Distribución final de 'host_is_superhost': 
host_is_superhost
False    6884
True     2830

[PASO 5] Binarizando e imputando 'host_response_time'...
 -> Nulos iniciales: 929. Nulos tras imputar moda agr

/tmp/ipykernel_16988/3871235952.py:163: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_name = df['host_name'].astype(str).str.contains(keywords, case=False, regex=True, na=False)


In [123]:
print("==================================================")
print("  INICIANDO PREPROCESAMIENTO: UBICACIÓN Y TEXTOS  ")
print("==================================================\n")

# =====================================================================
# 1. Añadir variable booleana 'neighbourhood_center'
# =====================================================================
print("[PASO 1] Creando variable 'neighbourhood_center' (Málaga Centro)...")

# Definimos expresiones regulares para capturar el centro de Málaga.
# Si tienes 'neighbourhood_group_cleansed', el distrito central en Málaga se llama simplemente "Centro".
# Añadimos también barrios específicos típicos del centro por si el grupo es nulo.
regex_centro = r'centro|histórico|historic|soho|merced|victoria|goleta|perchel'

# Evaluamos sobre el barrio específico
cond_barrio = df['neighbourhood_cleansed'].astype(str).str.contains(regex_centro, case=False, regex=True, na=False)

# Evaluamos sobre el distrito (grupo) si la columna existe en tu dataset
if 'neighbourhood_group_cleansed' in df.columns:
    cond_grupo = df['neighbourhood_group_cleansed'].astype(str).str.contains('centro', case=False, na=False)
    # Es del centro si el distrito es Centro O si el barrio específico es del centro
    df['neighbourhood_center'] = (cond_grupo | cond_barrio).astype(int)
else:
    df['neighbourhood_center'] = cond_barrio.astype(int)

anuncios_centro = df['neighbourhood_center'].sum()
porcentaje_centro = (anuncios_centro / len(df)) * 100

print(f" -> Alojamientos en el centro de Málaga (1): {anuncios_centro} ({porcentaje_centro:.2f}%)")
print(f" -> Alojamientos fuera del centro (0): {len(df) - anuncios_centro}\n")

# =====================================================================
# 2. Preprocesamiento PLN para 'neighborhood_overview'
# =====================================================================
print("[PASO 2] Limpiando textos de 'neighborhood_overview' para PLN...")

nulos_overview_antes = df['neighborhood_overview'].isnull().sum()

# Definimos la misma función de limpieza robusta (por si se ejecuta en una celda separada)
def clean_text_nlp(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)       # Quitar etiquetas HTML
    text = re.sub(r'[^\w\s]', ' ', text)       # Quitar signos de puntuación
    text = re.sub(r'\s+', ' ', text).strip()   # Reemplazar espacios múltiples por uno solo
    return text

df['neighborhood_overview_clean'] = df['neighborhood_overview'].apply(clean_text_nlp)

# Para no perder rastro de la ausencia de información (el texto vacío en PLN equivale a nada)
textos_vacios = (df['neighborhood_overview_clean'] == "").sum()

print(f" -> Textos nulos originales: {nulos_overview_antes}")
print(f" -> Textos vacíos tras la limpieza: {textos_vacios}")
if textos_vacios > 0:
    ejemplo_texto = df.loc[df['neighborhood_overview_clean'] != "", 'neighborhood_overview_clean'].iloc[0]
    print(f" -> Textos limpios con éxito. Ejemplo del primer registro válido:\n    '{ejemplo_texto[:60]}...'\n")

print("==================================================")
print("   PREPROCESAMIENTO DE UBICACIÓN FINALIZADO       ")
print("==================================================")

  INICIANDO PREPROCESAMIENTO: UBICACIÓN Y TEXTOS  

[PASO 1] Creando variable 'neighbourhood_center' (Málaga Centro)...
 -> Alojamientos en el centro de Málaga (1): 6389 (65.77%)
 -> Alojamientos fuera del centro (0): 3325

[PASO 2] Limpiando textos de 'neighborhood_overview' para PLN...
 -> Textos nulos originales: 5544
 -> Textos vacíos tras la limpieza: 5544
 -> Textos limpios con éxito. Ejemplo del primer registro válido:
    '200 metres from the beaches of el palo malaga s old fishing ...'

   PREPROCESAMIENTO DE UBICACIÓN FINALIZADO       


In [124]:
print("==================================================")
print(" INICIANDO PREPROCESAMIENTO: DISPONIBILIDAD E INGRESOS ")
print("==================================================\n")

# =====================================================================
# 1. Imputar valores nulos de 'has_availability'
# =====================================================================
print("[PASO 1] Imputando nulos de 'has_availability'...")
nulos_avail_antes = df['has_availability'].isnull().sum()

# Lógica: si tiene algún día disponible en los próximos 365 días, es True (1), si no False (0)
# Usamos availability_365 como el indicador más amplio posible.
dias_disponibles = df['availability_365'] > 0

df['has_availability'] = df['has_availability'].fillna(dias_disponibles)

# Por si originariamente venía como 't'/'f' como otras booleanas de Airbnb
df['has_availability'] = df['has_availability'].map({'t': True, 'f': False, True: True, False: False})

print(f" -> Nulos imputados: {nulos_avail_antes}")
print(f" -> Distribución final: True (Con disponibilidad) = {df['has_availability'].sum()} | False (Sin disponibilidad) = {(~df['has_availability']).sum()}\n")

# =====================================================================
# 2. Imputar 'estimated_revenue' (o estimated_revenue_l365d)
# =====================================================================
print("[PASO 2] Imputando 'estimated_revenue' usando la fórmula...")

# Comprobamos el nombre exacto de la columna en tu df
columna_revenue = 'estimated_revenue_l365d' if 'estimated_revenue_l365d' in df.columns else 'estimated_revenue'

if columna_revenue not in df.columns:
    df[columna_revenue] = np.nan # Si la habías borrado previamente, la recreamos

nulos_rev_antes = df[columna_revenue].isnull().sum()

# FÓRMULA DE INGRESOS ESTIMADOS:
# Si tenemos la ocupación estimada (estimated_occupancy_l365d), la multiplicamos por el precio.
# Si no, usamos la fórmula clásica: precio * días ocupados en el último año (365 - días disponibles).
if 'estimated_occupancy_l365d' in df.columns:
    ingresos_calculados = df['price'] * df['estimated_occupancy_l365d']
    print(" -> Fórmula utilizada: price * estimated_occupancy_l365d")
else:
    ingresos_calculados = df['price'] * (365 - df['availability_365'])
    print(" -> Fórmula utilizada: price * (365 - availability_365)")

# Imputamos los nulos con el cálculo
df[columna_revenue] = df[columna_revenue].fillna(ingresos_calculados)

# Aplicamos clip(lower=0) por si hubiera algún error en los datos que diera rentabilidad negativa
df[columna_revenue] = df[columna_revenue].clip(lower=0)

print(f" -> Valores nulos imputados/calculados: {nulos_rev_antes}")
print(f" -> Ingreso estimado medio del dataset: {df[columna_revenue].mean():.2f} €\n")

# =====================================================================
# 3. Transformación logarítmica de los ingresos
# =====================================================================
print("[PASO 3] Aplicando transformación logarítmica a los ingresos...")

# Usamos np.log1p (log(1 + x)) porque seguro que habrá ingresos = 0 
# (anuncios sin reservas) y el logaritmo de 0 daría error/infinito.
df['estimated_revenue_log'] = np.log1p(df[columna_revenue])

print(" -> Creada columna 'estimated_revenue_log'.")
print(f" -> Rango actual (escala logarítmica): Min {df['estimated_revenue_log'].min():.2f} - Max {df['estimated_revenue_log'].max():.2f}\n")

print("==================================================")
print(" PREPROCESAMIENTO DE DISPONIBILIDAD FINALIZADO    ")
print("==================================================")

 INICIANDO PREPROCESAMIENTO: DISPONIBILIDAD E INGRESOS 

[PASO 1] Imputando nulos de 'has_availability'...
 -> Nulos imputados: 88
 -> Distribución final: True (Con disponibilidad) = 9654 | False (Sin disponibilidad) = 60

[PASO 2] Imputando 'estimated_revenue' usando la fórmula...
 -> Fórmula utilizada: price * estimated_occupancy_l365d
 -> Valores nulos imputados/calculados: 899
 -> Ingreso estimado medio del dataset: 10683.00 €

[PASO 3] Aplicando transformación logarítmica a los ingresos...
 -> Creada columna 'estimated_revenue_log'.
 -> Rango actual (escala logarítmica): Min 0.00 - Max 15.50

 PREPROCESAMIENTO DE DISPONIBILIDAD FINALIZADO    


In [125]:
print("==================================================")
print("  INICIANDO PREPROCESAMIENTO: RESEÑAS Y FECHAS    ")
print("==================================================\n")

# =====================================================================
# 1. Añadir variable booleana 'has_review'
# =====================================================================
print("[PASO 1] Creando variable booleana 'has_review'...")

# Es True si el número total de reseñas es mayor que 0
df['has_review'] = (df['number_of_reviews'] > 0).astype(int)

con_resenas = df['has_review'].sum()
print(f" -> Alojamientos CON reseñas (1): {con_resenas} ({(con_resenas/len(df)*100):.2f}%)")
print(f" -> Alojamientos SIN reseñas (0): {len(df) - con_resenas}\n")

# =====================================================================
# 2. Añadir variable booleana 'has_review_ly' (Reseñas en 2025)
# =====================================================================
print("[PASO 2] Creando variable booleana 'has_review_ly' (Año 2025)...")

# Nos aseguramos de que last_review sea datetime para extraer el año
df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')

# Evaluamos si la última reseña fue en 2025 O si la variable de Airbnb number_of_reviews_ly es mayor a 0
cond_2025 = df['last_review'].dt.year == 2025
if 'number_of_reviews_ly' in df.columns:
    cond_ly = df['number_of_reviews_ly'] > 0
    df['has_review_ly'] = (cond_2025 | cond_ly).astype(int)
else:
    df['has_review_ly'] = cond_2025.astype(int)

print(f" -> Alojamientos activos con reseñas en el último año/2025: {df['has_review_ly'].sum()}\n")

# =====================================================================
# 3. Imputar valores nulos de 'reviews_per_month'
# =====================================================================
print("[PASO 3] Imputando valores nulos en 'reviews_per_month'...")

nulos_rpm_antes = df['reviews_per_month'].isnull().sum()

# Si no tienen reseñas (has_review == 0), rellenamos con 0
mask_sin_resenas = (df['has_review'] == 0)
df.loc[mask_sin_resenas, 'reviews_per_month'] = df.loc[mask_sin_resenas, 'reviews_per_month'].fillna(0)

# Por seguridad, si queda algún nulo residual (ej. un fallo del scrapeo de Airbnb), lo ponemos a 0
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

print(f" -> Nulos iniciales: {nulos_rpm_antes}")
print(f" -> Nulos tras imputar 0 a los alojamientos sin reseñas: {df['reviews_per_month'].isnull().sum()}\n")

# =====================================================================
# 4. Transformación de fechas a días transcurridos
# =====================================================================
print("[PASO 4] Transformando fechas de reseñas a días transcurridos...")

df['first_review'] = pd.to_datetime(df['first_review'], errors='coerce')

# Tomamos una fecha de referencia (puede ser la fecha actual o la de extracción del dataset)
# Por defecto cogemos la fecha máxima que aparezca en el dataset como punto de extracción
fecha_referencia = df['last_review'].max()
if pd.isna(fecha_referencia): 
    fecha_referencia = pd.Timestamp.today()

print(f" -> Fecha de referencia usada para el cálculo: {fecha_referencia.date()}")

# Calculamos los días transcurridos
df['days_since_first_review'] = (fecha_referencia - df['first_review']).dt.days
df['days_since_last_review'] = (fecha_referencia - df['last_review']).dt.days

# Los que no tienen reseñas tendrán valor nulo en los días, los rellenamos con 0 
# (el modelo los separará gracias a 'has_review')
df['days_since_first_review'] = df['days_since_first_review'].fillna(0)
df['days_since_last_review'] = df['days_since_last_review'].fillna(0)

print(f" -> Columnas 'days_since_first_review' y 'days_since_last_review' creadas e imputadas con 0 para nulos.\n")

# =====================================================================
# 5. Transformación logarítmica para variables de número de reseñas
# =====================================================================
print("[PASO 5] Aplicando transformación logarítmica (zipf/log) a cuentas de reseñas...")

cols_resenas = [
    'number_of_reviews', 
    'number_of_reviews_ltm', 
    'number_of_reviews_l30d', 
    'number_of_reviews_ly'
]

# Filtramos para asegurarnos de que la columna existe en el dataframe
cols_aplicar = [col for col in cols_resenas if col in df.columns]

for col in cols_aplicar:
    # Usamos np.log1p (log de 1+x) para manejar perfectamente los ceros
    df[f'{col}_log'] = np.log1p(df[col])

print(f" -> Transformación log1p aplicada a: {', '.join(cols_aplicar)}")
print(f" -> Ejemplo del nuevo rango para 'number_of_reviews_log': Min {df['number_of_reviews_log'].min():.2f} - Max {df['number_of_reviews_log'].max():.2f}\n")

print("==================================================")
print("   PREPROCESAMIENTO DE RESEÑAS FINALIZADO         ")
print("==================================================")

  INICIANDO PREPROCESAMIENTO: RESEÑAS Y FECHAS    

[PASO 1] Creando variable booleana 'has_review'...
 -> Alojamientos CON reseñas (1): 8709 (89.65%)
 -> Alojamientos SIN reseñas (0): 1005

[PASO 2] Creando variable booleana 'has_review_ly' (Año 2025)...
 -> Alojamientos activos con reseñas en el último año/2025: 8179

[PASO 3] Imputando valores nulos en 'reviews_per_month'...
 -> Nulos iniciales: 1005
 -> Nulos tras imputar 0 a los alojamientos sin reseñas: 0

[PASO 4] Transformando fechas de reseñas a días transcurridos...
 -> Fecha de referencia usada para el cálculo: 2025-09-30
 -> Columnas 'days_since_first_review' y 'days_since_last_review' creadas e imputadas con 0 para nulos.

[PASO 5] Aplicando transformación logarítmica (zipf/log) a cuentas de reseñas...
 -> Transformación log1p aplicada a: number_of_reviews, number_of_reviews_ltm, number_of_reviews_l30d, number_of_reviews_ly
 -> Ejemplo del nuevo rango para 'number_of_reviews_log': Min 0.00 - Max 7.01

   PREPROCESAMIENTO D

In [126]:
print("==================================================")
print("  INICIANDO PREPROCESAMIENTO: REGLAS DE RESERVA   ")
print("==================================================\n")

# =====================================================================
# 1. Transformación logarítmica de 'minimum_nights'
# =====================================================================
print("[PASO 1] Aplicando transformación logarítmica a 'minimum_nights'...")

# Usamos log1p por si hubiera algún valor anómalo de 0 noches
df['minimum_nights_log'] = np.log1p(df['minimum_nights'])

print(" -> Columna 'minimum_nights_log' generada con éxito.")
print(f" -> Rango original: Min {df['minimum_nights'].min()} - Max {df['minimum_nights'].max()}")
print(f" -> Rango logarítmico: Min {df['minimum_nights_log'].min():.2f} - Max {df['minimum_nights_log'].max():.2f}\n")

# =====================================================================
# 2. Variable booleana 'long_stay' (> 365 días)
# =====================================================================
print("[PASO 2] Creando variable booleana 'long_stay' (Estancias > 1 año)...")

# Detectamos a los que permiten estancias largas antes de aplicar el tope
df['long_stay'] = (df['maximum_nights'] > 365).astype(int)

print(f" -> Alojamientos que SÍ permiten alquiler tradicional/larga estancia (1): {df['long_stay'].sum()}")
print(f" -> Alojamientos puramente turísticos/corta estancia (0): {(df['long_stay'] == 0).sum()}\n")

# =====================================================================
# 3. Acotar 'maximum_nights' a 365 días (Capping)
# =====================================================================
print("[PASO 3] Acotando los valores de 'maximum_nights' a un tope de 365...")

max_original = df['maximum_nights'].max()

# .clip(upper=365) rebaja cualquier número superior (como 1125 o 1000) a 365
df['maximum_nights'] = df['maximum_nights'].clip(upper=365)

print(f" -> Valor máximo original detectado en el dataset: {max_original}")
print(f" -> Valor máximo tras aplicar el recorte (clipping): {df['maximum_nights'].max()}\n")

# =====================================================================
# 4. Convertir a booleana 'instant_bookable'
# =====================================================================
print("[PASO 4] Convirtiendo 'instant_bookable' a formato booleano...")

nulos_instant_antes = df['instant_bookable'].isnull().sum()

# Mapeamos los clásicos 't'/'f' de Airbnb a True/False nativos
df['instant_bookable'] = df['instant_bookable'].map({'t': True, 'f': False, True: True, False: False})

print(f" -> Nulos antes de convertir: {nulos_instant_antes}")
print(f" -> Distribución final: True (Reserva Inmediata) = {df['instant_bookable'].sum()} | False (Con aprobación) = {(~df['instant_bookable']).sum()}\n")

print("==================================================")
print("   PREPROCESAMIENTO DE RESERVAS FINALIZADO        ")
print("==================================================")

  INICIANDO PREPROCESAMIENTO: REGLAS DE RESERVA   

[PASO 1] Aplicando transformación logarítmica a 'minimum_nights'...
 -> Columna 'minimum_nights_log' generada con éxito.
 -> Rango original: Min 1 - Max 500
 -> Rango logarítmico: Min 0.69 - Max 6.22

[PASO 2] Creando variable booleana 'long_stay' (Estancias > 1 año)...
 -> Alojamientos que SÍ permiten alquiler tradicional/larga estancia (1): 2931
 -> Alojamientos puramente turísticos/corta estancia (0): 6783

[PASO 3] Acotando los valores de 'maximum_nights' a un tope de 365...
 -> Valor máximo original detectado en el dataset: 1125
 -> Valor máximo tras aplicar el recorte (clipping): 365

[PASO 4] Convirtiendo 'instant_bookable' a formato booleano...
 -> Nulos antes de convertir: 0
 -> Distribución final: True (Reserva Inmediata) = 6255 | False (Con aprobación) = 3459

   PREPROCESAMIENTO DE RESERVAS FINALIZADO        


In [127]:
print("==================================================")
print("  INICIANDO PREPROCESAMIENTO: LIMPIEZA FINAL Y RENOMBRADO  ")
print("==================================================\n")

# =====================================================================
# 1. Renombrar variables de Host
# =====================================================================
print("[PASO 1] Renombrando variables de Host...")

renombres = {
    'host_response_time': 'host_response_hour',
    'calculated_host_listings_count': 'host_listings_count',
    'calculated_host_listings_count_entire_homes': 'host_listings_count_eh',
    'calculated_host_listings_count_private_rooms': 'host_listings_count_pr',
    'calculated_host_listings_count_shared_rooms': 'host_listings_count_sr',
    'calculated_host_listings_hotel_rooms': 'host_listings_count_hr'
}

# Renombramos solo las que existan en el df para evitar errores
renombres_aplicar = {k: v for k, v in renombres.items() if k in df.columns}
df = df.rename(columns=renombres_aplicar)
print(f" -> Variables renombradas: {list(renombres_aplicar.values())}\n")

# =====================================================================
# 2. Eliminación masiva de columnas innecesarias
# =====================================================================
print("[PASO 2] Eliminando columnas descartadas para el estudio...")

cols_to_drop = [
    # Alojamiento
    'property_type', 'bathrooms_text', 'bathrooms_text_num', 'scrape_id', 'source', 'picture_url', 
    'license', 'last_scraped', 'calendar_last_scraped', 'calendar_updated',
    # Host
    'host_listings_count', 'host_about', 'host_url', 'host_picture_url', 
    'host_thumbnail_url', 'host_neighbourhood', 'host_response_hour'
    # Ubicación
    'neighborhood_overview', 'neighbourhood', 'neighbourhood_group_cleansed',
    # Disponibilidad (Redundantes)
    'availability_60', 'availability_90',
    # Reglas de Reserva (Históricas)
    'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 
    'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm'
]

# Filtramos para borrar solo las que aún sigan en el dataframe
cols_existentes = [col for col in cols_to_drop if col in df.columns]
df = df.drop(columns=cols_existentes)

print(f" -> Se han eliminado {len(cols_existentes)} columnas.")
print(f" -> Columnas actuales en el dataset: {len(df.columns)}\n")


print("==================================================")
print("       INICIANDO TASK: VALIDACIÓN DE DATOS        ")
print("==================================================\n")

filas_antes = len(df)

# =====================================================================
# Validación Alojamiento y Ubicación
# =====================================================================
print("[VALIDACIÓN] Comprobando Alojamiento y Ubicación...")

# Accommodates >= 1 y métricas positivas
cond_alojamiento = (
    (df['accommodates'] >= 1) & 
    (df['bathrooms'] >= 0) & 
    (df['bedrooms'] >= 0) & 
    (df['beds'] >= 0) & 
    (df['num_amenities'] >= 0)
)

# Latitud y longitud aproximadas de Málaga provincia (aprox: Lat 36.3 a 37.3, Lon -5.5 a -3.8)
cond_ubicacion = (
    df['latitude'].between(36.3, 37.3) & 
    df['longitude'].between(-5.5, -3.8)
)

# =====================================================================
# Validación Disponibilidad y Reseñas
# =====================================================================
print("[VALIDACIÓN] Comprobando Disponibilidad, Reseñas y Reglas...")

cond_disponibilidad = (
    df['availability_30'].between(0, 30) & 
    df['availability_365'].between(0, 365)
)

# Ratings entre 0 y 5 (ignorando los nulos con .fillna(True) para no borrar los que no tienen reseña)
cond_resenas = df['review_scores_rating'].between(0, 5) | df['review_scores_rating'].isna()

# Fechas de reseñas anteriores al scraping (2025-09-30)
fecha_limite = pd.to_datetime('2025-09-30')
cond_fechas = (df['last_review'] <= fecha_limite) | df['last_review'].isna()

# Reglas: máximo >= mínimo
cond_reglas = df['maximum_nights'] >= df['minimum_nights']

# =====================================================================
# APLICAR FILTROS Y LIMPIAR NULOS RESIDUALES
# =====================================================================
print("[VALIDACIÓN] Aplicando reglas y filtrando el dataset...")

# Juntamos todas las máscaras de validación
mascara_final = cond_alojamiento & cond_ubicacion & cond_disponibilidad & cond_resenas & cond_fechas & cond_reglas

df_validado = df[mascara_final].copy()
filas_invalidas = filas_antes - len(df_validado)
print(f" -> Filas eliminadas por no pasar las validaciones de rango lógico: {filas_invalidas}")

# Dropna final para asegurar el "No hay valores nulos" exigido en tu lista
# (Opcional: puedes restringir el dropna a un subset de columnas obligatorias)
cols_obligatorias = ['id', 'name', 'host_id']
nulos_antes_drop = len(df_validado)
df_validado = df_validado.dropna(subset=cols_obligatorias)
print(f" -> Filas eliminadas por nulos en campos clave: {nulos_antes_drop - len(df_validado)}")

# Validar IDs únicos
duplicados = df_validado.duplicated(subset=['id']).sum()
if duplicados > 0:
    df_validado = df_validado.drop_duplicates(subset=['id'])
    print(f" -> Filas eliminadas por ID duplicado: {duplicados}")

# =====================================================================
# DIAGNÓSTICO: ¿Por qué fallan las filas?
# =====================================================================
print("\n[DIAGNÓSTICO] Analizando las filas que van a ser eliminadas...")

# 1. Capturamos las filas que NO cumplen la máscara final
filas_erroneas = df[~mascara_final]

# 2. Si hay filas erróneas, analizamos una por una
if not filas_erroneas.empty:
    for indice, fila in filas_erroneas.iterrows():
        print(f"\n--- Anuncio ID: {fila['id']} ---")
        
        # Comprobamos condición por condición para este índice específico
        if not cond_alojamiento.loc[indice]: 
            print("❌ FALLO ALOJAMIENTO: Capacidad < 1 o valores negativos en camas/baños/amenities.")
        
        if not cond_ubicacion.loc[indice]: 
            print(f"❌ FALLO UBICACIÓN: Coordenadas raras. Lat: {fila['latitude']}, Lon: {fila['longitude']}")
        
        if not cond_disponibilidad.loc[indice]: 
            print("❌ FALLO DISPONIBILIDAD: availability_30 o availability_365 fuera de rango.")
            
        if not cond_resenas.loc[indice]: 
            print(f"❌ FALLO RESEÑAS: Rating fuera del rango 0-5. Valor actual: {fila['review_scores_rating']}")
            
        if not cond_fechas.loc[indice]: 
            print(f"❌ FALLO FECHAS: La última reseña es futura (mayor a 2025-09-30). Valor: {fila['last_review']}")
            
        if not cond_reglas.loc[indice]: 
            print(f"❌ FALLO REGLAS RESERVA: maximum_nights es MENOR que minimum_nights. Min: {fila['minimum_nights']}, Max: {fila['maximum_nights']}")
else:
    print("¡Todas las filas pasaron la validación con éxito!")

print(f"\n -> TAMAÑO FINAL DEL DATASET LISTO PARA KAFKA: {df_validado.shape}")
print("==================================================")
print("            VALIDACIÓN FINALIZADA                 ")
print("==================================================")

  INICIANDO PREPROCESAMIENTO: LIMPIEZA FINAL Y RENOMBRADO  

[PASO 1] Renombrando variables de Host...
 -> Variables renombradas: ['host_response_hour', 'host_listings_count', 'host_listings_count_eh', 'host_listings_count_pr', 'host_listings_count_sr', 'host_listings_count_hr']

[PASO 2] Eliminando columnas descartadas para el estudio...
 -> Se han eliminado 26 columnas.
 -> Columnas actuales en el dataset: 81

       INICIANDO TASK: VALIDACIÓN DE DATOS        

[VALIDACIÓN] Comprobando Alojamiento y Ubicación...
[VALIDACIÓN] Comprobando Disponibilidad, Reseñas y Reglas...
[VALIDACIÓN] Aplicando reglas y filtrando el dataset...
 -> Filas eliminadas por no pasar las validaciones de rango lógico: 2
 -> Filas eliminadas por nulos en campos clave: 0

[DIAGNÓSTICO] Analizando las filas que van a ser eliminadas...

--- Anuncio ID: 33862197 ---
❌ FALLO REGLAS RESERVA: maximum_nights es MENOR que minimum_nights. Min: 500, Max: 365

--- Anuncio ID: 34811272 ---
❌ FALLO REGLAS RESERVA: maximum_

In [128]:
for i in sorted(df_validado.columns):
    print(i)

accommodates
amenities
availability_30
availability_365
availability_eoy
bathroom_shared
bathrooms
bedrooms
beds
company
days_since_first_review
days_since_last_review
description
estimated_occupancy_l365d
estimated_revenue_l365d
estimated_revenue_log
first_review
has_availability
has_review
has_review_ly
host_acceptance_rate
host_has_profile_pic
host_id
host_identity_verified
host_is_superhost
host_listings_count_eh
host_listings_count_hr
host_listings_count_pr
host_listings_count_sr
host_location
host_name
host_near
host_response_hour
host_response_rate
host_response_time_bin
host_since
host_total_listings_count
host_verifications
id
instant_bookable
last_review
latitude
listing_url
long_stay
longitude
maximum_nights
minimum_nights
minimum_nights_log
name
neighborhood_overview
neighborhood_overview_clean
neighbourhood_center
neighbourhood_cleansed
num_amenities
number_of_reviews
number_of_reviews_l30d
number_of_reviews_l30d_log
number_of_reviews_log
number_of_reviews_ltm
number_of_re